# ARPD: Adaptive Evidence Retrieval with Paraphrase-Robust Training
## Section 0 — Setup & Session Restore
> Chạy section này ĐẦU TIÊN mỗi khi mở notebook

In [ ]:
# Cell 0.1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# Cell 0.2 — Upload src/ files (chỉ cần làm 1 lần)
# Chọn TẤT CẢ file .py trong thư mục src/ trên máy local
from google.colab import files
import os, shutil

os.makedirs('/content/src', exist_ok=True)
uploaded = files.upload()

for fname in uploaded.keys():
    shutil.copy(fname, f'/content/src/{fname}')
    shutil.copy(fname, f'/content/{fname}')
    print(f'✅ {fname}')

# Fix relative imports trong pipeline.py
with open('/content/src/pipeline.py', 'r') as f:
    content = f.read()
for mod in ['uncertainty_scorer','adaptive_retriever',
            'paraphrase_augmentor','encoder','classifier','data_loader']:
    content = content.replace(f'from .{mod}', f'from {mod}')
with open('/content/src/pipeline.py', 'w') as f:
    f.write(content)
with open('/content/pipeline.py', 'w') as f:
    f.write(content)
print('\n✅ All src files ready!')

In [ ]:
# Cell 0.3 — Install dependencies
!pip install sentence-transformers transformers torch \
             scikit-learn nltk wikipedia-api datasets scipy -q
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('✅ Dependencies ready')

In [ ]:
# Cell 0.4 — Import tất cả thư viện
import os, sys, pickle, shutil, zipfile, json, warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats
from collections import Counter
warnings.filterwarnings('ignore')

sys.path.insert(0, '/content')
sys.path.insert(0, '/content/src')

from data_loader import load_liar
from paraphrase_augmentor import synonym_substitute, combined_augment

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
print('✅ All imports successful')

In [ ]:
# Cell 0.5 — Define core classes và functions

class ImprovedMLP(nn.Module):
    """Improved MLP với BatchNorm, Residual connection, Xavier init."""
    def __init__(self, input_dim=384):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.4)
        )
        self.block2 = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.block3 = nn.Sequential(
            nn.Linear(128, 64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(0.2)
        )
        self.classifier = nn.Linear(64, 2)
        self.residual   = nn.Linear(input_dim, 128)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        res = self.residual(x)
        out = self.block1(x)
        out = self.block2(out) + res
        out = self.block3(out)
        return self.classifier(out)


class SimpleMLP(nn.Module):
    def __init__(self, input_dim=384):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x): return self.net(x)


def encode_pairs(claims, evidences, encoder):
    combined = [f'{c} [SEP] {e}' if e else c
                for c, e in zip(claims, evidences)]
    return encoder.encode(combined, batch_size=32, show_progress_bar=False)

def add_full_context(df):
    return [f"[{row['speaker']}] [{row['subject']}] {row['claim']}"
            for _, row in df.iterrows()]

def train_mlp(model, X_train, y_train, class_weights, device,
              epochs=25, lr=2e-4, seed=42, patience=6, label_smoothing=0.1):
    """Train MLP với warmup scheduler, grad clipping, early stopping."""
    torch.manual_seed(seed)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights, label_smoothing=label_smoothing
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    full_ds  = TensorDataset(torch.FloatTensor(X_train),
                              torch.LongTensor(y_train))
    val_size = int(0.1 * len(full_ds))
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(
        full_ds, [len(full_ds)-val_size, val_size], generator=g
    )
    loader_tr  = DataLoader(train_ds, batch_size=64, shuffle=True)
    loader_val = DataLoader(val_ds,   batch_size=64)

    total_steps  = len(loader_tr) * epochs
    warmup_steps = total_steps // 10
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, 1-(step-warmup_steps)/(total_steps-warmup_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val, patience_counter, step = float('inf'), 0, 0
    best_path = f'/content/best_mlp_{seed}.pt'

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader_tr:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); step += 1

        model.eval()
        val_loss = sum(
            criterion(model(xb.to(device)), yb.to(device)).item()
            for xb, yb in loader_val
        ) / len(loader_val)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), best_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(torch.load(best_path, map_location=device))
    return model


def ensemble_predict(model, tfidf_model, lr_model, X_emb, X_tfidf,
                     device, w_arpd=0.15):
    """Ensemble prediction: (1-w)*TF-IDF + w*MLP."""
    model.eval()
    with torch.no_grad():
        prob_mlp = torch.softmax(
            model(torch.FloatTensor(X_emb).to(device)), dim=1
        ).cpu().numpy()
    prob_lr = lr_model.predict_proba(X_tfidf)
    return (1-w_arpd)*prob_lr + w_arpd*prob_mlp


def evaluate(labels, preds):
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_fake':  f1_score(labels, preds, pos_label=0),
        'f1_real':  f1_score(labels, preds, pos_label=1),
    }

print('✅ All classes and functions defined')

In [ ]:
# Cell 0.6 — Load LIAR dataset và cached evidences
import os, shutil, zipfile, pickle

# Restore LIAR từ Drive
os.makedirs('/content/data/raw', exist_ok=True)
shutil.copy('/content/drive/MyDrive/ARPD-research/liar_dataset.zip',
            '/content/data/raw/liar_dataset.zip')
with zipfile.ZipFile('/content/data/raw/liar_dataset.zip', 'r') as z:
    z.extractall('/content/data/raw/')

train_df = load_liar(split='train',      cache_dir='/content/data/raw')
val_df   = load_liar(split='validation', cache_dir='/content/data/raw')
test_df  = load_liar(split='test',       cache_dir='/content/data/raw')
print(f'✅ LIAR: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}')

# Load cached evidences
with open('/content/drive/MyDrive/ARPD-research/results/train_evidences.pkl','rb') as f:
    train_evidences = pickle.load(f)
with open('/content/drive/MyDrive/ARPD-research/results/test_evidences.pkl','rb') as f:
    test_evidences = pickle.load(f)
print(f'✅ Evidences: train={len(train_evidences):,}, test={len(test_evidences):,}')

In [ ]:
# Cell 0.7 — Encode claims + Build TF-IDF
from sentence_transformers import SentenceTransformer

model_st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Encoding LIAR claims...')

train_claims = add_full_context(train_df)
test_claims  = add_full_context(test_df)

X_train_emb = encode_pairs(train_claims, train_evidences, model_st)
X_test_emb  = encode_pairs(test_claims,  test_evidences,  model_st)
print(f'✅ Embeddings: train={X_train_emb.shape}, test={X_test_emb.shape}')

# TF-IDF
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,3))
X_train_tfidf = tfidf.fit_transform(train_claims)
X_test_tfidf  = tfidf.transform(test_claims)

# Class weights
cw_vals = compute_class_weight('balanced',
    classes=np.array([0,1]), y=train_df['label'].values)
class_weights = torch.FloatTensor(cw_vals).to(device)
print(f'✅ Class weights: FAKE={cw_vals[0]:.3f}, REAL={cw_vals[1]:.3f}')
print('\n🚀 Setup complete! Ready to run experiments.')

---
## Section 1 — Baselines
> Chạy 3 baselines để có số so sánh

In [ ]:
# Cell 1.1 — Baseline 1: TF-IDF + Logistic Regression
lr_base = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
lr_base.fit(X_train_tfidf, train_df['label'])
preds_lr = lr_base.predict(X_test_tfidf)
res_lr = evaluate(test_df['label'], preds_lr)

print('BASELINE 1: TF-IDF + Logistic Regression')
print(f"  Accuracy:  {res_lr['accuracy']:.4f}")
print(f"  F1-Macro:  {res_lr['f1_macro']:.4f}")
print(f"  F1-FAKE:   {res_lr['f1_fake']:.4f}")
print(f"  F1-REAL:   {res_lr['f1_real']:.4f}")

In [ ]:
# Cell 1.2 — Baseline 2: TF-IDF + SVM
svm = CalibratedClassifierCV(LinearSVC(max_iter=2000, C=1.0))
svm.fit(X_train_tfidf, train_df['label'])
preds_svm = svm.predict(X_test_tfidf)
res_svm = evaluate(test_df['label'], preds_svm)

print('BASELINE 2: TF-IDF + SVM')
print(f"  Accuracy:  {res_svm['accuracy']:.4f}")
print(f"  F1-Macro:  {res_svm['f1_macro']:.4f}")
print(f"  F1-FAKE:   {res_svm['f1_fake']:.4f}")
print(f"  F1-REAL:   {res_svm['f1_real']:.4f}")

In [ ]:
# Cell 1.3 — Baseline 3: MiniLM + SimpleMLP (no retrieval)
mlp_base = SimpleMLP().to(device)
mlp_base = train_mlp(mlp_base, X_train_emb, train_df['label'].tolist(),
                      class_weights, device, epochs=20, seed=42)
mlp_base.eval()
with torch.no_grad():
    preds_mlp = mlp_base(
        torch.FloatTensor(X_test_emb).to(device)
    ).argmax(1).cpu().numpy()
res_mlp = evaluate(test_df['label'], preds_mlp)

print('BASELINE 3: MiniLM + SimpleMLP')
print(f"  Accuracy:  {res_mlp['accuracy']:.4f}")
print(f"  F1-Macro:  {res_mlp['f1_macro']:.4f}")
print(f"  F1-FAKE:   {res_mlp['f1_fake']:.4f}")
print(f"  F1-REAL:   {res_mlp['f1_real']:.4f}")

---
## Section 2 — ARPD Improved Pipeline
> Train ARPD với ImprovedMLP + Ensemble

In [ ]:
# Cell 2.1 — Train ARPD ImprovedMLP (seed=42)
torch.manual_seed(42)
np.random.seed(42)

mlp_arpd = ImprovedMLP(input_dim=384).to(device)
total_params = sum(p.numel() for p in mlp_arpd.parameters())
print(f'ImprovedMLP params: {total_params:,}')

mlp_arpd = train_mlp(mlp_arpd, X_train_emb, train_df['label'].tolist(),
                      class_weights, device, epochs=25, seed=42)
print('✅ ImprovedMLP trained')

In [ ]:
# Cell 2.2 — Grid search ensemble weights
print('Grid search ensemble weights...')
best_acc, best_w, best_preds = 0, 0.15, None

for w in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35]:
    prob = ensemble_predict(mlp_arpd, tfidf, lr_base,
                            X_test_emb, X_test_tfidf, device, w)
    preds = prob.argmax(1)
    acc   = accuracy_score(test_df['label'], preds)
    f1    = f1_score(test_df['label'], preds, average='macro')
    mark  = ' <- best' if acc > best_acc else ''
    print(f'  w={w:.2f}: Acc={acc:.4f}, F1={f1:.4f}{mark}')
    if acc > best_acc:
        best_acc, best_w, best_preds = acc, w, preds

res_arpd = evaluate(test_df['label'], best_preds)
print(f'\n✅ Best weight: {best_w}')
print(f"  Accuracy:  {res_arpd['accuracy']:.4f}")
print(f"  F1-Macro:  {res_arpd['f1_macro']:.4f}")
print(f"  F1-FAKE:   {res_arpd['f1_fake']:.4f}")
print(f"  F1-REAL:   {res_arpd['f1_real']:.4f}")
print(f'\n{classification_report(test_df["label"], best_preds, target_names=["FAKE","REAL"])}')

---
## Section 3 — Statistical Significance Test (5 seeds)
> BẮT BUỘC cho journal quốc tế

In [ ]:
# Cell 3.1 — Run 5 seeds
from scipy import stats as scipy_stats

SEEDS = [42, 123, 456, 789, 2025]
sig_results = {'arpd': [], 'lr': []}

for i, seed in enumerate(SEEDS):
    print(f'\n--- Seed {seed} ({i+1}/{len(SEEDS)}) ---')
    torch.manual_seed(seed); np.random.seed(seed)

    # Train ImprovedMLP
    m = ImprovedMLP(input_dim=384).to(device)
    m = train_mlp(m, X_train_emb, train_df['label'].tolist(),
                  class_weights, device, epochs=25, seed=seed)

    # Grid search weight
    best_acc_s, best_w_s, best_p_s = 0, 0.15, None
    prob_lr_s = lr_base.predict_proba(X_test_tfidf)
    m.eval()
    with torch.no_grad():
        prob_m_s = torch.softmax(
            m(torch.FloatTensor(X_test_emb).to(device)), dim=1
        ).cpu().numpy()
    for w in [0.10, 0.15, 0.20, 0.25, 0.30]:
        p = (1-w)*prob_lr_s + w*prob_m_s
        a = accuracy_score(test_df['label'], p.argmax(1))
        if a > best_acc_s:
            best_acc_s, best_w_s, best_p_s = a, w, p

    preds_s = best_p_s.argmax(1)
    acc_s  = accuracy_score(test_df['label'], preds_s)
    f1_s   = f1_score(test_df['label'], preds_s, average='macro')
    f1_f_s = f1_score(test_df['label'], preds_s, pos_label=0)

    # TF-IDF+LR với seed
    lr_s = LogisticRegression(max_iter=1000, C=0.5, random_state=seed)
    lr_s.fit(X_train_tfidf, train_df['label'])
    preds_lr_s = lr_s.predict(X_test_tfidf)
    acc_lr_s = accuracy_score(test_df['label'], preds_lr_s)
    f1_lr_s  = f1_score(test_df['label'], preds_lr_s, average='macro')

    sig_results['arpd'].append({'acc': acc_s, 'f1': f1_s, 'f1_fake': f1_f_s})
    sig_results['lr'].append({'acc': acc_lr_s, 'f1': f1_lr_s})
    print(f'  ARPD (w={best_w_s}): Acc={acc_s:.4f}, F1={f1_s:.4f}, F1-FAKE={f1_f_s:.4f}')
    print(f'  TF-IDF+LR:          Acc={acc_lr_s:.4f}, F1={f1_lr_s:.4f}')

# Tổng hợp
arpd_accs = [r['acc']     for r in sig_results['arpd']]
arpd_f1s  = [r['f1']      for r in sig_results['arpd']]
arpd_ffs  = [r['f1_fake'] for r in sig_results['arpd']]
lr_accs   = [r['acc']     for r in sig_results['lr']]
lr_f1s    = [r['f1']      for r in sig_results['lr']]

t_acc, p_acc = scipy_stats.ttest_rel(arpd_accs, lr_accs)
t_f1,  p_f1  = scipy_stats.ttest_rel(arpd_f1s,  lr_f1s)

print(f'\n{"="*65}')
print('STATISTICAL SIGNIFICANCE (5 seeds)')
print(f'{"="*65}')
print(f'{"Model":<22} {"Accuracy":>16} {"F1-Macro":>16} {"F1-FAKE":>14}')
print(f'{"-"*65}')
print(f'{"TF-IDF+LR":<22} '
      f'{np.mean(lr_accs):>6.4f}+-{np.std(lr_accs):.4f}  '
      f'{np.mean(lr_f1s):>6.4f}+-{np.std(lr_f1s):.4f}  {"—":>14}')
print(f'{"ARPD (Improved)":<22} '
      f'{np.mean(arpd_accs):>6.4f}+-{np.std(arpd_accs):.4f}  '
      f'{np.mean(arpd_f1s):>6.4f}+-{np.std(arpd_f1s):.4f}  '
      f'{np.mean(arpd_ffs):>6.4f}+-{np.std(arpd_ffs):.4f}')
print(f'\nPaired t-test (ARPD vs TF-IDF+LR):')
print(f'  Accuracy: t={t_acc:.4f}, p={p_acc:.4f} '
      f'{"SIGNIFICANT" if p_acc<0.05 else "Not significant"}')
print(f'  F1-Macro: t={t_f1:.4f}, p={p_f1:.4f} '
      f'{"SIGNIFICANT" if p_f1<0.05 else "Not significant"}')

---
## Section 4 — Robustness Evaluation
> Paraphrase attack p=0.50 trên full test set

In [ ]:
# Cell 4.1 — Robustness test (5 seeds)
rob_arpd, rob_lr = [], []

for i, seed in enumerate(SEEDS):
    print(f'Seed {seed}...')
    torch.manual_seed(seed); np.random.seed(seed)

    # Paraphrase test claims
    para_claims = [synonym_substitute(r['claim'], p=0.5, seed=seed)
                   for _, r in test_df.iterrows()]
    para_ctx = [f"[{r['speaker']}] [{r['subject']}] {p}"
                for (_, r), p in zip(test_df.iterrows(), para_claims)]
    X_para_emb   = encode_pairs(para_ctx, test_evidences, model_st)
    X_para_tfidf = tfidf.transform(para_ctx)

    # Load best ARPD model cho seed này
    m = ImprovedMLP(input_dim=384).to(device)
    m.load_state_dict(torch.load(f'/content/best_mlp_{seed}.pt', map_location=device))
    m.eval()

    # ARPD clean vs attacked
    prob_lr_c = lr_base.predict_proba(X_test_tfidf)
    with torch.no_grad():
        prob_m_c = torch.softmax(
            m(torch.FloatTensor(X_test_emb).to(device)), dim=1
        ).cpu().numpy()
    acc_c = accuracy_score(test_df['label'],
                           (0.85*prob_lr_c + 0.15*prob_m_c).argmax(1))

    prob_lr_p = lr_base.predict_proba(X_para_tfidf)
    with torch.no_grad():
        prob_m_p = torch.softmax(
            m(torch.FloatTensor(X_para_emb).to(device)), dim=1
        ).cpu().numpy()
    acc_p = accuracy_score(test_df['label'],
                           (0.85*prob_lr_p + 0.15*prob_m_p).argmax(1))
    rob_arpd.append(acc_c - acc_p)

    # TF-IDF+LR robustness
    lr_s = LogisticRegression(max_iter=1000, C=0.5, random_state=seed)
    lr_s.fit(X_train_tfidf, train_df['label'])
    acc_lr_c = accuracy_score(test_df['label'], lr_s.predict(X_test_tfidf))
    acc_lr_p = accuracy_score(test_df['label'], lr_s.predict(X_para_tfidf))
    rob_lr.append(acc_lr_c - acc_lr_p)

    print(f'  ARPD drop={rob_arpd[-1]:+.4f} | LR drop={rob_lr[-1]:+.4f}')

t_rob, p_rob = scipy_stats.ttest_rel(rob_lr, rob_arpd)
print(f'\nROBUSTNESS DROP (lower = better):')
print(f'  TF-IDF+LR: {np.mean(rob_lr):.4f} +- {np.std(rob_lr):.4f}')
print(f'  ARPD:      {np.mean(rob_arpd):.4f} +- {np.std(rob_arpd):.4f}')
print(f'  t={t_rob:.4f}, p={p_rob:.4f} '
      f'{"ARPD significantly more robust!" if p_rob<0.05 else "Not significant"}')

---
## Section 5 — Ablation Study
> Chứng minh contribution của từng component

In [ ]:
# Cell 5.1 — Ablation Study
print('Running Ablation Study...')
ablation = {}

# Full ARPD
ablation['Full ARPD'] = res_arpd

# No retrieval (claim only, no evidence)
X_train_noret = model_st.encode(train_claims, batch_size=32, show_progress_bar=False)
X_test_noret  = model_st.encode(test_claims,  batch_size=32, show_progress_bar=False)
m_noret = ImprovedMLP(input_dim=384).to(device)
m_noret = train_mlp(m_noret, X_train_noret, train_df['label'].tolist(),
                    class_weights, device, epochs=20, seed=42)
m_noret.eval()
with torch.no_grad():
    prob_noret = torch.softmax(
        m_noret(torch.FloatTensor(X_test_noret).to(device)), dim=1
    ).cpu().numpy()
prob_lr_base = lr_base.predict_proba(X_test_tfidf)
preds_noret = (0.85*prob_lr_base + 0.15*prob_noret).argmax(1)
ablation['No Retrieval'] = evaluate(test_df['label'], preds_noret)
print('  No Retrieval done')

# No speaker context
train_claims_nos = [row['claim'] for _, row in train_df.iterrows()]
test_claims_nos  = [row['claim'] for _, row in test_df.iterrows()]
X_train_nos = encode_pairs(train_claims_nos, train_evidences, model_st)
X_test_nos  = encode_pairs(test_claims_nos,  test_evidences,  model_st)
tfidf_nos = TfidfVectorizer(max_features=15000, ngram_range=(1,3))
X_tr_tfidf_nos = tfidf_nos.fit_transform(train_claims_nos)
X_te_tfidf_nos = tfidf_nos.transform(test_claims_nos)
lr_nos = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
lr_nos.fit(X_tr_tfidf_nos, train_df['label'])
m_nos = ImprovedMLP(input_dim=384).to(device)
m_nos = train_mlp(m_nos, X_train_nos, train_df['label'].tolist(),
                  class_weights, device, epochs=20, seed=42)
m_nos.eval()
with torch.no_grad():
    prob_nos_m = torch.softmax(
        m_nos(torch.FloatTensor(X_test_nos).to(device)), dim=1
    ).cpu().numpy()
prob_nos_lr = lr_nos.predict_proba(X_te_tfidf_nos)
preds_nos = (0.85*prob_nos_lr + 0.15*prob_nos_m).argmax(1)
ablation['No Speaker'] = evaluate(test_df['label'], preds_nos)
print('  No Speaker Context done')

# No ensemble (MLP only)
mlp_arpd.eval()
with torch.no_grad():
    preds_noens = mlp_arpd(
        torch.FloatTensor(X_test_emb).to(device)
    ).argmax(1).cpu().numpy()
ablation['No Ensemble'] = evaluate(test_df['label'], preds_noens)
print('  No Ensemble done')

# Print results
print(f'\n{"="*65}')
print('ABLATION STUDY')
print(f'{"="*65}')
print(f'{"Variant":<22} {"Accuracy":>10} {"F1-Macro":>10} {"Drop":>10}')
print(f'{"-"*55}')
full_acc = ablation['Full ARPD']['accuracy']
for name, r in ablation.items():
    drop = f"{r['accuracy']-full_acc:+.4f}" if name != 'Full ARPD' else '—'
    print(f"{name:<22} {r['accuracy']:>10.4f} {r['f1_macro']:>10.4f} {drop:>10}")

---
## Section 6 — FEVER Dataset Experiments
> Cross-domain validation

In [ ]:
# Cell 6.1 — Load FEVER từ Drive
fever_train_df = pd.read_csv(
    '/content/drive/MyDrive/ARPD-research/fever_train_full.csv'
)
fever_test_df  = pd.read_csv(
    '/content/drive/MyDrive/ARPD-research/fever_test_full.csv'
)
print(f'FEVER Train: {len(fever_train_df):,}')
print(f'FEVER Test:  {len(fever_test_df):,}')
ev_rate = (fever_test_df['evidence'].fillna('') != '').mean()
print(f'Evidence coverage: {ev_rate*100:.1f}%')

In [ ]:
# Cell 6.2 — Balance + Encode FEVER
from sklearn.utils import resample

# Balance
fever_real = fever_train_df[fever_train_df['label']==1]
fever_fake = fever_train_df[fever_train_df['label']==0]
fever_real_bal = resample(fever_real, n_samples=len(fever_fake), random_state=42)
fever_train_bal = pd.concat([fever_real_bal, fever_fake]).sample(frac=1, random_state=42)
fever_train_sub = fever_train_bal.head(10000).copy()
fever_test_sub  = fever_test_df.head(5000).copy()
print(f'Balanced train: {len(fever_train_sub):,} | Test: {len(fever_test_sub):,}')

# Encode
def encode_fever(df, encoder):
    combined = [
        f"{row['claim']} [SEP] {row['evidence']}"
        if str(row.get('evidence','')) not in ['','nan'] else row['claim']
        for _, row in df.iterrows()
    ]
    return encoder.encode(combined, batch_size=64, show_progress_bar=False)

print('Encoding FEVER...')
X_fever_train_emb = encode_fever(fever_train_sub, model_st)
X_fever_test_emb  = encode_fever(fever_test_sub,  model_st)

tfidf_fever = TfidfVectorizer(max_features=15000, ngram_range=(1,3))
X_fever_tr_tfidf = tfidf_fever.fit_transform(fever_train_sub['claim'])
X_fever_te_tfidf = tfidf_fever.transform(fever_test_sub['claim'])

cw_fever = compute_class_weight('balanced',
    classes=np.array([0,1]), y=fever_train_sub['label'].values)
cw_fever_t = torch.FloatTensor(cw_fever).to(device)
print('✅ FEVER encoded')

In [ ]:
# Cell 6.3 — Train và Evaluate ARPD trên FEVER

# Baseline
lr_fever = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
lr_fever.fit(X_fever_tr_tfidf, fever_train_sub['label'])
res_lr_fever = evaluate(fever_test_sub['label'],
                        lr_fever.predict(X_fever_te_tfidf))

# ARPD
mlp_fever = ImprovedMLP(input_dim=384).to(device)
mlp_fever = train_mlp(mlp_fever, X_fever_train_emb,
                      fever_train_sub['label'].tolist(),
                      cw_fever_t, device, epochs=20, seed=42)

# Grid search
best_acc_f, best_w_f, best_preds_f = 0, 0.25, None
prob_lr_f = lr_fever.predict_proba(X_fever_te_tfidf)
mlp_fever.eval()
with torch.no_grad():
    prob_m_f = torch.softmax(
        mlp_fever(torch.FloatTensor(X_fever_test_emb).to(device)), dim=1
    ).cpu().numpy()
for w in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]:
    p = (1-w)*prob_lr_f + w*prob_m_f
    a = accuracy_score(fever_test_sub['label'], p.argmax(1))
    if a > best_acc_f:
        best_acc_f, best_w_f = a, w
        best_preds_f = p.argmax(1)

res_arpd_fever = evaluate(fever_test_sub['label'], best_preds_f)

print('FEVER RESULTS:')
print(f'{"Model":<20} {"Accuracy":>10} {"F1-Macro":>10} {"F1-FAKE":>10}')
print('-'*52)
print(f'{"TF-IDF+LR":<20} '
      f"{res_lr_fever['accuracy']:>10.4f} "
      f"{res_lr_fever['f1_macro']:>10.4f} "
      f"{res_lr_fever['f1_fake']:>10.4f}")
print(f'{"ARPD (FEVER)":<20} '
      f"{res_arpd_fever['accuracy']:>10.4f} "
      f"{res_arpd_fever['f1_macro']:>10.4f} "
      f"{res_arpd_fever['f1_fake']:>10.4f}")

---
## Section 7 — Export All Results
> Lưu tất cả kết quả ra CSV và JSON

In [ ]:
# Cell 7.1 — Export tất cả kết quả
import json
from datetime import datetime

all_results = {
    'timestamp': datetime.now().isoformat(),
    'dataset': 'LIAR',
    'baselines': {
        'tfidf_lr':  res_lr,
        'tfidf_svm': res_svm,
        'minilm_mlp': res_mlp,
    },
    'arpd': res_arpd,
    'statistical_significance': {
        'arpd_acc_mean': float(np.mean(arpd_accs)),
        'arpd_acc_std':  float(np.std(arpd_accs)),
        'arpd_f1_mean':  float(np.mean(arpd_f1s)),
        'arpd_f1_std':   float(np.std(arpd_f1s)),
        'lr_acc_mean':   float(np.mean(lr_accs)),
        'lr_f1_mean':    float(np.mean(lr_f1s)),
        'p_value_acc':   float(p_acc),
        'p_value_f1':    float(p_f1),
    },
    'robustness': {
        'arpd_drop_mean': float(np.mean(rob_arpd)),
        'arpd_drop_std':  float(np.std(rob_arpd)),
        'lr_drop_mean':   float(np.mean(rob_lr)),
        'lr_drop_std':    float(np.std(rob_lr)),
        'p_value':        float(p_rob),
    },
    'ablation': {k: v for k, v in ablation.items()},
    'fever': {
        'tfidf_lr': res_lr_fever,
        'arpd':     res_arpd_fever,
    }
}

# Save JSON
json_path = '/content/drive/MyDrive/ARPD-research/results/FINAL_RESULTS.json'
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'✅ Results saved: {json_path}')

# Summary table
print(f'\n{"="*70}')
print('FINAL SUMMARY TABLE')
print(f'{"="*70}')
print(f'{"Model":<25} {"Dataset":>8} {"Acc":>8} {"F1":>8} {"F1-FAKE":>8}')
print(f'{"-"*60}')
rows = [
    ('TF-IDF+LR',     'LIAR',  res_lr['accuracy'],        res_lr['f1_macro'],        res_lr['f1_fake']),
    ('TF-IDF+SVM',    'LIAR',  res_svm['accuracy'],       res_svm['f1_macro'],       res_svm['f1_fake']),
    ('MiniLM+MLP',    'LIAR',  res_mlp['accuracy'],       res_mlp['f1_macro'],       res_mlp['f1_fake']),
    ('ARPD (Improved)','LIAR', res_arpd['accuracy'],      res_arpd['f1_macro'],      res_arpd['f1_fake']),
    ('TF-IDF+LR',     'FEVER', res_lr_fever['accuracy'],  res_lr_fever['f1_macro'],  res_lr_fever['f1_fake']),
    ('ARPD (Improved)','FEVER',res_arpd_fever['accuracy'],res_arpd_fever['f1_macro'],res_arpd_fever['f1_fake']),
]
for name, ds, acc, f1, ff in rows:
    print(f'{name:<25} {ds:>8} {acc:>8.4f} {f1:>8.4f} {ff:>8.4f}')
print(f'\n✅ All experiments complete!')